In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn as nn

from tqdm import tqdm

from sklearn.metrics import auc, average_precision_score, precision_recall_curve, roc_auc_score, roc_curve
from torch.utils.data import DataLoader


In [2]:
PROJECT_DIR = Path("./")
sys.path.insert(0, str(PROJECT_DIR.resolve()))

TRAIN_DIR = PROJECT_DIR / "Dataset_train_no_process"
VAL_DIR = PROJECT_DIR / "Dataset_validation_no_process"
TRAIN_XLSX = PROJECT_DIR / "train.xlsx"
VAL_XLSX = PROJECT_DIR / "validation.xlsx"
OUTPUT_DIR = PROJECT_DIR / "history_3d_simple_cnn_no_process"

label_columns = ["ICH"]
target_size = 128

epoch = 18
batch_size_train = 8
batch_size_val = 8
num_workers = 4


In [3]:
from dataset_no_process import CTVolumeDataset


class ConvBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout_p=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout3d(p=dropout_p) if dropout_p > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.block(x)


class Simple3DClassifier(nn.Module):
    def __init__(self, in_channels=1, num_classes=1):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock3D(in_channels, 16, stride=2, dropout_p=0.1),
            ConvBlock3D(16, 32, stride=2, dropout_p=0.1),
            ConvBlock3D(32, 64, stride=2, dropout_p=0.1),
            ConvBlock3D(64, 128, stride=2, dropout_p=0.1),
            ConvBlock3D(128, 256, stride=2, dropout_p=0.1),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.head(x)
        return x


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_dataset = CTVolumeDataset(
    xlsx_path=TRAIN_XLSX,
    images_dir=TRAIN_DIR,
    label_columns=label_columns,
    target_size=target_size,
)

val_dataset = CTVolumeDataset(
    xlsx_path=VAL_XLSX,
    images_dir=VAL_DIR,
    label_columns=label_columns,
    target_size=target_size,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size_train,
    shuffle=False,
    num_workers=num_workers,
    prefetch_factor = 2,
    pin_memory=(device.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size_val,
    shuffle=False,
    num_workers=num_workers,
    prefetch_factor = 2,
    pin_memory=(device.type == "cuda"),
)

model = Simple3DClassifier(in_channels=1, num_classes=len(label_columns)).to(device)
checkpoint = torch.load(OUTPUT_DIR / f"checkpoint_epoch_{epoch}.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


Using device: cuda


Simple3DClassifier(
  (encoder): Sequential(
    (0): ConvBlock3D(
      (block): Sequential(
        (0): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
        (1): InstanceNorm3d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (2): ReLU(inplace=True)
        (3): Dropout3d(p=0.1, inplace=False)
      )
    )
    (1): ConvBlock3D(
      (block): Sequential(
        (0): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
        (1): InstanceNorm3d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (2): ReLU(inplace=True)
        (3): Dropout3d(p=0.1, inplace=False)
      )
    )
    (2): ConvBlock3D(
      (block): Sequential(
        (0): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
        (1): InstanceNorm3d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (2): ReLU(inplace=True)
 

In [7]:
next(model.parameters()).device

device(type='cuda', index=0)

In [8]:
criterion = nn.BCEWithLogitsLoss()

def collect_split_predictions(model, loader, device):
    targets = []
    scores = []

    loss_sum = 0.0

    with torch.no_grad():
        for volumes, labels in tqdm(loader, leave=False):
            volumes = volumes.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(volumes)

            loss = criterion(logits, labels)
            loss_sum += loss.item()

            probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)

            scores.append(probs)
            targets.append(labels.cpu().numpy().reshape(-1))

    mean_loss = round(loss_sum / max(len(loader), 1), 5)
    print('Mean loss', mean_loss)
    y_true = np.concatenate(targets, axis=0)
    y_score = np.concatenate(scores, axis=0)
    return y_true, y_score


train_y_true, train_y_score = collect_split_predictions(
    model=model,
    loader=train_loader,
    device=device,
)

val_y_true, val_y_score = collect_split_predictions(
    model=model,
    loader=val_loader,
    device=device,
)


Mean loss 0.12488


Mean loss 0.52002


In [9]:
def plot_preliminary_plots(y_true, y_score, dataset_name):
    # Добавляем width и height в гистограмму
    fig_hist = px.histogram(x=y_score, color=y_true.astype(str), nbins=50,
                            labels={'color': 'True Labels', 'x': 'Score'},
                            title=f'{dataset_name}: Histogram of Scores',
                            width=800,   # ← ДОБАВИТЬ
                            height=500)  # ← ДОБАВИТЬ
    fig_hist.show()

    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    df_thresh = pd.DataFrame({'FPR': fpr, 'TPR': tpr}, index=thresholds)
    
    # Добавляем width и height в график TPR/FPR
    fig_thresh = px.line(df_thresh, 
                         title=f'{dataset_name}: TPR and FPR at every threshold',
                         width=800,    # ← ДОБАВИТЬ
                         height=500)   # ← ДОБАВИТЬ
    fig_thresh.update_yaxes(scaleanchor="x", scaleratio=1)
    fig_thresh.update_xaxes(range=[0, 1], constrain='domain')
    fig_thresh.show()

def plot_roc_curve(y_true, y_score, dataset_name):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    print(f'ROC AUC: {roc_auc:.3f}')
    # Добавляем width и height в ROC кривую
    fig = px.area(x=fpr, y=tpr, 
                  title=f'{dataset_name}: ROC Curve (AUC={roc_auc:.3f})',
                  labels={'x': 'False Positive Rate', 'y': 'True Positive Rate'},
                  width=800,    # ← ДОБАВИТЬ
                  height=500)   # ← ДОБАВИТЬ
    fig.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=0, y1=1)
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.show()

def plot_pr_curve(y_true, y_score, dataset_name):
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    pr_auc = auc(recall, precision)
    print(f'PR AUC: {pr_auc:.3f}')
    # Добавляем width и height в PR кривую
    fig = px.area(x=recall, y=precision, 
                  title=f'{dataset_name}: Precision-Recall Curve (AUC={pr_auc:.3f})',
                  labels={'x': 'Recall', 'y': 'Precision'},
                  width=800,    # ← ДОБАВИТЬ
                  height=500)   # ← ДОБАВИТЬ
    fig.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=1, y1=0)
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.show()

In [11]:
for y_true, y_score, name in [(train_y_true, train_y_score, 'Train'),
                               (val_y_true, val_y_score, 'Validation')]:

# for y_true, y_score, name in [(train_y_true, train_y_score, 'Train')]:
    plot_preliminary_plots(y_true, y_score, name)
    plot_roc_curve(y_true, y_score, name)
    plot_pr_curve(y_true, y_score, name)

ROC AUC: 1.000


PR AUC: 1.000


ROC AUC: 0.794


PR AUC: 0.733


In [11]:
import pandas as pd

pd.Series(train_y_score).to_csv('train_predictions_no_process.csv', index=False, header=True)
pd.Series(val_y_score).to_csv('validation_predictions_no_process.csv', index=False, header=True)
